# Séance 7 — Régression en pratique I

**Analyse des données — L3 Économie**

On construit le modèle **une variable à la fois**, en regardant à chaque étape ce que les coefficients deviennent.

> *Exécution → Tout exécuter* avant de démarrer.


## 0. Les données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm03-salaires.csv")

df = pd.read_csv(URL, sep=";", encoding="utf-8", decimal=",", thousands=" ")

# le nettoyage de la seance 3
df["SALAIRE_NET"] = df["SALAIRE_NET"].replace(999999, np.nan)
df["DIPLOME"]     = df["DIPLOME"].replace(9, np.nan)
df["TEMPS"]       = df["TEMPS"].replace(9, np.nan)
df = df[df["SALAIRE_NET"] > 0].copy()

# les variables de Mincer
ANNEES = {1: 7, 2: 9, 3: 11, 4: 12, 5: 14, 6: 15, 7: 17}
df["ETUDE"] = df["DIPLOME"].map(ANNEES)
df["EXPER"] = df["AGE"] - df["ETUDE"] - 6
df["FEMME"] = (df["SEXE"] == 2).astype(int)
df["LSAL"]  = np.log(df["SALAIRE_NET"])

d = df.dropna(subset=["LSAL", "ETUDE", "EXPER"])
d = d[d["EXPER"] >= 0].copy()
print(len(d), "observations")

## 1. Une variable, une droite

La régression cherche la droite qui minimise la somme des **carrés** des écarts verticaux.

Les losanges sont les moyennes de salaire à chaque niveau d'études. La droite ne les relie pas : elle en est l'**ajustement pondéré par les effectifs**. Elle résume sept nombres en un seul, et ce résumé a un coût — voir la cellule suivante.


In [ ]:
m1 = smf.ols("LSAL ~ ETUDE", data=d).fit(cov_type="HC1")

fig, ax = plt.subplots(figsize=(7, 3.6))
rng = np.random.default_rng(0)
ax.scatter(d["ETUDE"] + rng.normal(0, .22, len(d)), d["LSAL"],
           s=3, alpha=.12, color="#9AA0A0", linewidths=0)

xs = np.linspace(7, 17, 50)
ax.plot(xs, m1.params["Intercept"] + m1.params["ETUDE"] * xs, color="#E2162F", lw=2)

moy = d.groupby("ETUDE")["LSAL"].mean()
ax.scatter(moy.index, moy.values, s=55, color="#3D4543", marker="D", zorder=5)

ax.set_xlabel("Années d'études")
ax.set_ylabel("log(salaire net mensuel)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# la droite passe-t-elle vraiment par les moyennes ?
moy = d.groupby("ETUDE")["LSAL"].agg(["mean", "count"])
moy["droite"] = m1.params["Intercept"] + m1.params["ETUDE"] * moy.index
moy["ecart"]  = moy["mean"] - moy["droite"]
moy["en %"]   = 100 * (np.exp(moy["ecart"]) - 1)
print(moy.round(4).to_string())

Non : l'écart atteint 0,078 au niveau bac+3/4, soit **8 % de salaire**. Invisible sur la figure, mais économiquement réel.

La droite est l'ajustement des moindres carrés sur ces sept moyennes, **pondéré par les effectifs** : le niveau à 15 années pèse 797 observations contre 1 512 à 11 années, et la droite penche vers les groupes les plus nombreux. Une régression pondérée faite sur les sept losanges seuls redonne exactement les mêmes coefficients, 6,954809 et 0,063832.

Elle ne passerait par chaque losange que dans un modèle **saturé**, avec une muette par niveau — le `C(DIPLOME)` de la section 7. Imposer une droite, c'est faire l'hypothèse que le rendement est le même à chaque palier.

''## 2. Lire la sortie

Les quatre nombres d'une ligne :

| colonne | ce qu'elle dit |
|---|---|
| `coef` | de combien la variable expliquée bouge pour une unité de plus |
| `std err` | l'incertitude sur ce nombre — seule, elle ne se commente pas |
| `P>\|z\|` | la probabilité d'un coefficient aussi éloigné de zéro si le vrai était nul |
| `[0.025 0.975]` | la plage de valeurs plausibles — c'est elle qu'on cite |

En haut de la sortie, deux lignes à vérifier systématiquement : `No. Observations` (a-t-on perdu des lignes sans le vouloir ?) et `Covariance Type` (quels écarts-types ?).


In [ ]:
print(m1.summary())

## 3. Rappel — ce que teste un $t$

On part d'une **hypothèse nulle** : le vrai coefficient vaut zéro. On ne cherche pas à la prouver, mais à voir si les données la rendent invraisemblable.

Le $t$ est le coefficient divisé par son écart-type : **de combien d'écarts-types le coefficient s'éloigne de zéro.**

Trois façons de dire la même chose :

$$|t| > 1{,}96 \quad \Longleftrightarrow \quad p < 0{,}05 \quad \Longleftrightarrow \quad \text{l'intervalle de confiance ne contient pas zéro}$$


In [ ]:
# le t n'a rien de mysterieux : c'est une division
coef = m1.params["ETUDE"]
se   = m1.bse["ETUDE"]

print("coefficient      ", round(coef, 5))
print("ecart-type       ", round(se, 5))
print("coef / ecart-type", round(coef / se, 3))
print("t donne par statsmodels", round(m1.tvalues["ETUDE"], 3))
print()
bas, haut = m1.conf_int().loc["ETUDE"]
print(f"intervalle a 95 % : coef +/- 1.96 * se = "
      f"[{coef - 1.96 * se:.4f}, {coef + 1.96 * se:.4f}]")
print(f"intervalle de statsmodels              = [{bas:.4f}, {haut:.4f}]")

In [ ]:
# a quoi ressemble un t qui ne serait PAS significatif ?
# on fabrique une variable sans aucun lien avec le salaire
d["BRUIT"] = rng.normal(size=len(d))
faux = smf.ols("LSAL ~ BRUIT", data=d).fit(cov_type="HC1")

print(faux.summary().tables[1])

**Les deux phrases à ne jamais écrire.**

« $p$ est la probabilité que l'hypothèse nulle soit vraie » — non : $p$ se calcule *en supposant* qu'elle l'est.

« Ce n'est pas significatif, donc l'effet est nul » — non : l'absence de preuve n'est pas la preuve de l'absence.

## 4. On ajoute l'expérience


In [ ]:
m2 = smf.ols("LSAL ~ ETUDE + EXPER + I(EXPER**2)", data=d).fit(cov_type="HC1")

print("ETUDE seule           ", round(m1.params["ETUDE"], 4), "  R2 =", round(m1.rsquared, 3))
print("ETUDE avec experience ", round(m2.params["ETUDE"], 4), "  R2 =", round(m2.rsquared, 3))
print()
print("correlation ETUDE / EXPER :", round(d["ETUDE"].corr(d["EXPER"]), 3))
print()
print(m2.params.round(5).to_string())

Le rendement de l'éducation **monte** : 0,0638 → 0,0716.

À âge donné, celui qui a étudié plus longtemps a travaillé moins longtemps. Les deux variables sont corrélées négativement **par construction**, et la première régression mélangeait donc deux effets de signe opposé.

**La leçon : un coefficient n'a de sens qu'avec la liste de ce qui est tenu constant à côté.**

Le coefficient du carré est négatif : le profil est **concave**. Chaque année d'expérience rapporte un peu moins que la précédente.

## 5. On ajoute une muette

`FEMME` ne vaut que 0 ou 1. La droite n'a donc que deux endroits où se poser — et elle passe exactement par la moyenne de chaque groupe.


In [ ]:
print(d.groupby("FEMME")["LSAL"].mean().round(6).to_string())

m0 = smf.ols("LSAL ~ FEMME", data=d).fit(cov_type="HC1")
print()
print("constante   ", round(m0.params["Intercept"], 6))
print("coefficient ", round(m0.params["FEMME"], 6))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.scatter(d["FEMME"] + rng.normal(0, .035, len(d)), d["LSAL"],
           s=3, alpha=.10, color="#9AA0A0", linewidths=0)

mh = d.loc[d["FEMME"] == 0, "LSAL"].mean()
mf = d.loc[d["FEMME"] == 1, "LSAL"].mean()
ax.plot([0, 1], [mh, mf], color="#E2162F", lw=2.4)
ax.scatter([0, 1], [mh, mf], s=90, color="#3D4543", marker="D", zorder=5)

ax.set_xticks([0, 1]); ax.set_xticklabels(["FEMME = 0", "FEMME = 1"])
ax.set_ylabel("log(salaire net mensuel)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 6. Le même nombre, obtenu deux fois

Le test de la séance 5 et la régression d'aujourd'hui donnent le **même** $t$, à la quatrième décimale. Deux fois, selon l'hypothèse faite sur les variances.


In [ ]:
h = d.loc[d["FEMME"] == 0, "LSAL"]
f = d.loc[d["FEMME"] == 1, "LSAL"]
m0s = smf.ols("LSAL ~ FEMME", data=d).fit()      # ecarts-types par defaut

print("variances egales : Student", round(stats.ttest_ind(f, h, equal_var=True).statistic, 4),
      "| regression", round(m0s.tvalues["FEMME"], 4))
print("variances libres : Welch  ", round(stats.ttest_ind(f, h, equal_var=False).statistic, 4),
      "| regression", round(m0.tvalues["FEMME"], 4))

**Le test de Welch était un écart-type robuste.** Tout ce que vous avez appris en séance 5 est un cas particulier de la régression.

## 7. Plus de deux modalités

Avec $k$ modalités, `statsmodels` crée $k-1$ muettes. La modalité omise devient la **référence**, logée dans la constante ; chaque coefficient se lit par rapport à elle.

`C(...)` force le traitement en qualitatif. Sans lui, `DIPLOME` serait pris pour un nombre.


In [ ]:
m6 = smf.ols("LSAL ~ C(DIPLOME) + EXPER + I(EXPER**2) + FEMME",
             data=d).fit(cov_type="HC1")

noms = {2: "BEPC", 3: "CAP-BEP", 4: "bac", 5: "bac+2", 6: "bac+3/4", 7: "bac+5"}
print("reference : aucun diplome\n")
for k, lab in noms.items():
    print(f"   {lab:<9} {m6.params[f'C(DIPLOME)[T.{k}.0]']: .4f}")

## 8. Le modèle complet — une phrase par coefficient

In [ ]:
m3 = smf.ols("LSAL ~ ETUDE + EXPER + I(EXPER**2) + FEMME",
             data=d).fit(cov_type="HC1")
print(m3.summary())

In [ ]:
# un coefficient en logarithme se traduit en pourcentage
for v in ["ETUDE", "EXPER", "FEMME"]:
    print(f"{v:<6} coef {m3.params[v]: .4f}   soit "
          f"{100 * (np.exp(m3.params[v]) - 1): .1f} % de salaire")

- `ETUDE` — une année d'études de plus est associée à **7,5 %** de salaire en plus, à expérience et sexe donnés.
- `EXPER` et son carré — une année d'expérience rapporte **1,6 %** en début de carrière, et le gain s'atténue ensuite.
- `FEMME` — à éducation et expérience identiques, les femmes gagnent **14,1 %** de moins.
- `Intercept` — un homme sans études ni expérience : une extrapolation qu'on ne commente pas.

Un $R^2$ de 0,218 sur données individuelles est normal. Un $R^2$ de 0,95 signalerait une variable qui contient déjà la réponse.

## 9. Les écarts-types robustes

`cov_type` ne change **que** l'incertitude. Le coefficient ne bouge jamais.


In [ ]:
for formule in ["LSAL ~ ETUDE", "SALAIRE_NET ~ ETUDE"]:
    a = smf.ols(formule, data=d).fit()
    b = smf.ols(formule, data=d).fit(cov_type="HC1")
    print(f"{formule:<22} coef {a.params['ETUDE']:9.4f}"
          f"   se {a.bse['ETUDE']:8.4f}   seHC1 {b.bse['ETUDE']:8.4f}")

En logarithme, l'écart entre les deux est de 1 %. En euros, il atteint 5 % : la dispersion des salaires grandit avec le niveau d'études en montant, beaucoup moins en pourcentage. Le logarithme absorbe l'essentiel de l'hétéroscédasticité.

**La règle :** `cov_type="HC1"` par défaut. Ce n'est pas de le mettre qui se justifie.

## 10. Ce qu'un coefficient ne dit pas

> Une année d'études supplémentaire est **associée** à un salaire supérieur d'environ 7,5 %, à expérience et sexe donnés.

Et non : « fait gagner 7,5 % ». Ceux qui étudient plus longtemps diffèrent des autres par bien plus que leur diplôme, et rien de cela n'est dans le fichier.

**TD 7 — l'équation de Mincer.** Une phrase par coefficient.
